# RT-DETR-L: 5-Fold Weighted Boxes Fusion Ensemble

## Overview

This notebook implements our final object detection pipeline using five independently trained RT-DETR-L models.

Instead of relying on a single checkpoint, we combine predictions from all five cross-validation folds using **Weighted Boxes Fusion (WBF)** to improve detection accuracy.

## Methodology

- **Architecture:** RT-DETR-L (query-based object detector)
- **Models:** Five checkpoints trained using video-level GroupKFold
- **Input resolution:** 960 × 960
- **Confidence threshold:** 0.01
- **WBF IoU threshold:** 0.55
- **WBF minimum score:** 0.02
- **Ensemble strategies:**
  - Equal WBF: `[1, 1, 1, 1, 1]`
  - Weighted WBF: `[1, 1, 1, 1, 2]` (double weight for Fold 4)

Overlapping predictions of the same class are combined through confidence-weighted averaging of bounding-box coordinates.

## Results

| Ensemble | Kaggle Public LB |
|---|---:|
| Equal WBF | 0.2882303 |
| **Weighted WBF** | **0.2898437** |

Weighted WBF achieved our highest public leaderboard score and was selected as our final submission.

## Reproducibility

The notebook uses pretrained checkpoints produced by `RT-DETR-L Tuned 5-Fold CV`.

Required dependencies include `ultralytics==8.4.202`, `ensemble-boxes==1.0.9`, PyTorch, OpenCV, NumPy and pandas.

The trained checkpoints and competition images are loaded from Kaggle Inputs and are not included in the GitHub repository.

Running the notebook generates two files:

- `submission.csv` — weighted ensemble.
- `submission_equal_wbf.csv` — equally weighted ensemble.

In [1]:

# ============================================================
# RT-DETR-L TUNED — 5-FOLD WBF ENSEMBLE
# Kaggle submission | NO TRAINING
# Two outputs from ONE GPU inference pass
# ============================================================

import sys
import subprocess
import time
import json
import importlib.metadata
from pathlib import Path

# 1. Install compatible package versions
NEEDED = {
    "ultralytics": "8.4.202",
    "ensemble-boxes": "1.0.9",
}

for package, version in NEEDED.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None

    if installed != version:
        print(f"Installing {package}=={version}")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "-q", f"{package}=={version}"
        ])

import cv2
import numpy as np
import pandas as pd
import torch

from ultralytics import RTDETR
from ensemble_boxes import weighted_boxes_fusion

# ============================================================
# 2. PATHS AND SETTINGS
# ============================================================

COMP = Path(
    "/kaggle/input/competitions/"
    "multiple-object-video-detection-hw-2-ucu-ai-2026"
)

FOLDS = Path(
    "/kaggle/input/notebooks/vikhimiak/"
    "rt-detr-l-tuned-5-fold-cv/"
    "rtdetr_tuned/cv_runs"
)

WORK = Path("/kaggle/working")
CHUNKS = WORK / "wbf_5fold_modelaware_chunks"
CHUNKS.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = [
    FOLDS / f"tuned_cv_fold{k}/weights/best.pt"
    for k in range(5)
]

EXPECTED_CLASSES = [
    "person", "bicycle", "car", "van",
    "truck", "tricycle", "bus", "motor"
]

PREDICT_ARGS = {
    "imgsz": 960,
    "conf": 0.01,
    "iou": 0.7,
    "max_det": 300,
    "device": 0,
    "verbose": False,
}

WBF_IOU = 0.55
WBF_MIN_SCORE = 0.02
OUTPUT_MAX_DET = 500
CHUNK_SIZE = 50

WEIGHTS_EQUAL = [1, 1, 1, 1, 1]
WEIGHTS_FOLD4 = [1, 1, 1, 1, 2]

# ============================================================
# 3. PRE-FLIGHT CHECKS
# ============================================================

assert torch.cuda.is_available(), "Enable GPU T4!"
assert COMP.is_dir(), "Competition data missing!"

missing_models = [
    str(p) for p in MODEL_PATHS if not p.is_file()
]
assert not missing_models, missing_models

test = pd.read_csv(
    COMP / "test_images.csv",
    dtype={"image_id": str, "file_name": str}
)

assert len(test) == 3811
assert test["image_id"].is_unique

assert {
    "image_id", "file_name", "width", "height"
}.issubset(test.columns)

missing_images = [
    str(COMP / f)
    for f in test["file_name"]
    if not (COMP / f).is_file()
]

assert not missing_images, missing_images[:5]

# Prevent accidental reuse of incompatible cached chunks
signature = {
    "models": [
        {"path": str(p), "bytes": p.stat().st_size}
        for p in MODEL_PATHS
    ],
    "predict": PREDICT_ARGS,
    "wbf_conf_type": "box_and_model_avg",
    "wbf_iou": WBF_IOU,
    "wbf_min_score": WBF_MIN_SCORE,
    "output_max_det": OUTPUT_MAX_DET,
    "chunk_size": CHUNK_SIZE,
    "weights_equal": WEIGHTS_EQUAL,
    "weights_fold4": WEIGHTS_FOLD4,
    "n_images": len(test),
}

config_path = CHUNKS / "config.json"

if config_path.exists():
    assert json.loads(config_path.read_text()) == signature, (
        "Cached chunks use different settings!"
    )
else:
    assert not list(CHUNKS.glob("*.csv")), (
        "Chunks exist without configuration!"
    )
    config_path.write_text(json.dumps(signature, indent=2))

# Test WBF before loading checkpoints
smoke = weighted_boxes_fusion(
    [
        [[0.1, 0.1, 0.2, 0.2]],
        [[0.1, 0.1, 0.2, 0.2]],
    ],
    [[0.9], [0.8]],
    [[0], [0]],
    weights=[1, 1],
    iou_thr=0.55,
    skip_box_thr=0.02,
)

assert len(smoke[0]) == 1

print("=" * 65)
print("RT-DETR-L TUNED — 5-FOLD WBF ENSEMBLE")
print("=" * 65)
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoints:", len(MODEL_PATHS))
print("Test images:", len(test))
print("WBF IoU:", WBF_IOU)
print("WBF minimum score:", WBF_MIN_SCORE)

# ============================================================
# 4. LOAD ALL FIVE TRAINED MODELS
# ============================================================

print("\nLoading 5 checkpoints...", flush=True)

models = [RTDETR(str(p)) for p in MODEL_PATHS]

for k, model in enumerate(models):
    names = [model.names[i] for i in range(8)]
    assert names == EXPECTED_CLASSES, (
        f"Fold {k} class mapping mismatch: {names}"
    )
    print(f"Fold {k}: MODEL LOADED")

print("\nAll five models verified!")

# ============================================================
# 5. CONVERT PREDICTIONS TO NORMALIZED WBF FORMAT
# ============================================================

def to_wbf(result, width, height):
    xyxy = (
        result.boxes.xyxy.detach()
        .cpu().numpy().astype(np.float64)
    )

    scores = (
        result.boxes.conf.detach()
        .cpu().numpy().astype(np.float64)
    )

    labels = (
        result.boxes.cls.detach()
        .cpu().numpy().astype(np.int64)
    )

    if not len(scores):
        return [], [], []

    xyxy[:, [0, 2]] /= width
    xyxy[:, [1, 3]] /= height

    np.clip(xyxy, 0.0, 1.0, out=xyxy)

    keep = (
        (scores >= WBF_MIN_SCORE)
        & np.isfinite(scores)
        & np.isfinite(xyxy).all(axis=1)
        & (xyxy[:, 2] > xyxy[:, 0])
        & (xyxy[:, 3] > xyxy[:, 1])
        & (labels >= 0)
        & (labels < 8)
    )

    return (
        xyxy[keep].tolist(),
        scores[keep].tolist(),
        labels[keep].tolist(),
    )

# ============================================================
# 6. WEIGHTED BOXES FUSION AND SUBMISSION FORMATTING
# ============================================================

def fuse_and_format(
    boxes_list,
    scores_list,
    labels_list,
    weights,
    width,
    height
):

    assert len(boxes_list) == len(scores_list) == len(labels_list) == len(weights) == 5

    boxes, scores, labels = weighted_boxes_fusion(
        boxes_list,
        scores_list,
        labels_list,
        weights=weights,
        iou_thr=WBF_IOU,
        skip_box_thr=WBF_MIN_SCORE,
        conf_type="box_and_model_avg",
        allows_overflow=False,
    )

    if len(scores) == 0:
        return ""

    # Keep at most 500 highest-confidence detections
    keep = np.argsort(
        -scores, kind="stable"
    )[:OUTPUT_MAX_DET]

    values = []

    for ix in keep:
        x1, y1, x2, y2 = boxes[ix]

        x1, x2 = sorted((
            float(x1) * width,
            float(x2) * width
        ))

        y1, y2 = sorted((
            float(y1) * height,
            float(y2) * height
        ))

        x1 = max(0.0, min(x1, width))
        x2 = max(0.0, min(x2, width))
        y1 = max(0.0, min(y1, height))
        y2 = max(0.0, min(y2, height))

        if x2 <= x1 or y2 <= y1:
            continue

        cls = int(labels[ix])
        score = float(scores[ix])

        assert 0 <= cls < 8
        assert np.isfinite(score) and score >= 0

        values.extend([
            str(cls),
            f"{min(score, 1.0):.6f}",
            f"{x1:.2f}",
            f"{y1:.2f}",
            f"{x2-x1:.2f}",
            f"{y2-y1:.2f}",
        ])

    return " ".join(values)

# Five models; deliberately duplicate boxes in Fold 4
test_boxes = [
    [],
    [],
    [],
    [],
    [[0.1, 0.1, 0.2, 0.2] for _ in range(6)],
]

test_scores = [
    [], [], [], [], [0.95] * 6
]

test_labels = [
    [], [], [], [], [0] * 6
]

for weights in (WEIGHTS_EQUAL, WEIGHTS_FOLD4):
    test_pred = fuse_and_format(
        test_boxes,
        test_scores,
        test_labels,
        weights,
        1000,
        1000,
    )

    assert test_pred
    parts = test_pred.split()

    assert len(parts) == 6
    assert 0 <= float(parts[1]) <= 1

    print(f"WBF TEST PASS | weights={weights} | score={parts[1]}")

# ============================================================
# 7. PROCESS TEST IMAGES IN RESUMABLE CHUNKS
# ============================================================

def checked_chunk(path, expected_ids):
    if not path.is_file():
        return False

    df = pd.read_csv(
        path, dtype={"image_id": str}
    ).fillna({"PredictionString": ""})

    return (
        df.columns.tolist()
        == ["image_id", "PredictionString"]
        and df["image_id"].tolist() == expected_ids
    )


start = time.perf_counter()
N = len(test)

num_chunks = (N + CHUNK_SIZE - 1) // CHUNK_SIZE

print("\n" + "=" * 65)
print("STARTING FIVE-MODEL INFERENCE")
print("=" * 65)

for part in range(num_chunks):
    begin = part * CHUNK_SIZE
    end = min(N, begin + CHUNK_SIZE)

    batch = test.iloc[begin:end]
    expected_ids = batch["image_id"].tolist()

    f_weighted = CHUNKS / f"weighted_{part:04d}.csv"
    f_equal = CHUNKS / f"equal_{part:04d}.csv"

    if (
        checked_chunk(f_weighted, expected_ids)
        and checked_chunk(f_equal, expected_ids)
    ):
        print(
            f"Chunk {part+1}/{num_chunks}: cached, skipping",
            flush=True
        )
        continue

    result_weighted = []
    result_equal = []

    for row in batch.itertuples(index=False):
        img_path = COMP / row.file_name

        # Read image ONCE, reuse for all five models
        img = cv2.imread(
            str(img_path), cv2.IMREAD_COLOR
        )

        assert img is not None, img_path

        h, w = img.shape[:2]

        assert h == int(row.height)
        assert w == int(row.width)

        boxes_list = []
        scores_list = []
        labels_list = []

        # Get predictions from all 5 trained models
        for model in models:
            res = model.predict(
                source=img,
                **PREDICT_ARGS
            )[0]

            assert res.orig_shape == (h, w)

            b, s, lab = to_wbf(res, w, h)

            boxes_list.append(b)
            scores_list.append(s)
            labels_list.append(lab)

        # Variant 1: Fold 4 gets more weight
        pred_weighted = fuse_and_format(
            boxes_list,
            scores_list,
            labels_list,
            WEIGHTS_FOLD4,
            w, h
        )

        # Variant 2: Equal weights
        pred_equal = fuse_and_format(
            boxes_list,
            scores_list,
            labels_list,
            WEIGHTS_EQUAL,
            w, h
        )

        result_weighted.append({
            "image_id": row.image_id,
            "PredictionString": pred_weighted,
        })

        result_equal.append({
            "image_id": row.image_id,
            "PredictionString": pred_equal,
        })

    # Save both chunks atomically
    for out, rows in [
        (f_weighted, result_weighted),
        (f_equal, result_equal),
    ]:
        tmp = out.with_suffix(".csv.tmp")

        pd.DataFrame(
            rows,
            columns=["image_id", "PredictionString"]
        ).to_csv(tmp, index=False)

        tmp.replace(out)

    elapsed = (time.perf_counter() - start) / 60

    eta = (
        elapsed / end * (N - end)
        if end else 0
    )

    print(
        f"Chunk {part+1}/{num_chunks}"
        f" | {end}/{N} images"
        f" | elapsed {elapsed:.1f} min"
        f" | estimated remaining {eta:.1f} min",
        flush=True,
    )

# ============================================================
# 8. COMBINE AND VALIDATE BOTH SUBMISSIONS
# ============================================================

print("\nCombining completed chunks...")

for name, prefix in [
    ("submission.csv", "weighted"),
    ("submission_equal_wbf.csv", "equal"),
]:
    frames = []

    for part in range(num_chunks):
        path = CHUNKS / f"{prefix}_{part:04d}.csv"

        assert path.is_file(), path

        frames.append(
            pd.read_csv(
                path,
                dtype={"image_id": str}
            ).fillna({"PredictionString": ""})
        )

    df = pd.concat(frames, ignore_index=True)

    assert df.columns.tolist() == [
        "image_id", "PredictionString"
    ]

    assert len(df) == 3811

    assert (
        df["image_id"].tolist()
        == test["image_id"].tolist()
    )

    total_boxes = 0
    empty_images = 0

    for pred in df["PredictionString"]:
        pred = str(pred)

        if not pred:
            empty_images += 1
            continue

        tokens = pred.split()

        assert len(tokens) % 6 == 0

        count = len(tokens) // 6

        assert 0 < count <= OUTPUT_MAX_DET

        total_boxes += count

    output_path = WORK / name
    tmp = output_path.with_suffix(".csv.tmp")

    df.to_csv(tmp, index=False)
    tmp.replace(output_path)

    print("\nFILE:", name)
    print("Rows:", len(df))
    print("Detections:", total_boxes)
    print("Empty images:", empty_images)
    print(
        "Size:",
        round(output_path.stat().st_size / 1024**2, 2),
        "MB"
    )

# ============================================================
# 9. FINISHED
# ============================================================

elapsed = (time.perf_counter() - start) / 60

print("\n" + "=" * 65)
print("RT-DETR-L TUNED — 5-FOLD ENSEMBLE COMPLETE")
print("=" * 65)

print("Total time:", round(elapsed, 2), "minutes")
print("Models:", 5)
print("Test images:", N)

print("\nOUTPUT FILES:")
print("/kaggle/working/submission.csv")
print("/kaggle/working/submission_equal_wbf.csv")

print("\nALL VALIDATION CHECKS PASSED!")
print("NO TRAINING WAS PERFORMED!")


Installing ultralytics==8.4.202
Installing ensemble-boxes==1.0.9
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
RT-DETR-L TUNED — 5-FOLD WBF ENSEMBLE
GPU: Tesla T4
Checkpoints: 5
Test images: 3811
WBF IoU: 0.55
WBF minimum score: 0.02

Loading 5 checkpoints...
Fold 0: MODEL LOADED
Fold 1: MODEL LOADED
Fold 2: MODEL LOADED
Fold 3: MODEL LOADED
Fold 4: MODEL LOADED

All five models verified!
WBF TEST PASS | weights=[1, 1, 1, 1, 1] | score=0.190000
WBF TEST PASS | weights=[1, 1, 1, 1, 2] | score=0.316667

STARTING FIVE-MODEL INFERENCE
Chunk 1/77 | 50/3811 images | elapsed 0.9 min | estimated remaining 65.5 min
Chunk 2/77 | 100/3811 images | elapsed 1.6 min | estimated remaining 57.6 min
Chunk 3/77 | 150/3811 images | elapsed 2.3 min | estimated rem